# Altı bariyer · eğitim → doğrulama → kilit → test

2/5, 1/1, 1/2, 1.5/2, 2/2 ve 3/2 ATR; 10 saat ufuk; aynı mumda çift temas: zarar önce.
Altı temel özellik, GRU, BCE, wavelet kapalı. Üç seed, toplam 18 eğitim.
Bu yeni tek dönem çalışmasıdır; eski 38-fold sonuçlarıyla birleştirilmez.

| Bölüm | UTC dönem | Görev |
|---|---|---|
| Eğitim | 18 Aralık 2025–15 Temmuz 2026 | Ağırlık ve scaler öğrenimi |
| Doğrulama | 17 Temmuz–8 Ağustos 2026 | En düşük BCE ile epoch seçimi |
| Eşik doğrulaması | 10–30 Ağustos 2026 | Ortalama F1 ile ortak aday eşiği |
| Test | 2–14 Ekim 2026 | Kilitli 6 aday, 312 saat; seçim yapılmaz |

Ağustos geçmişte görülmüştür; burada geliştirme verisidir. Eylül test edilmez.
Test verisi yalnızca tüm modeller/eşikler kilitlendikten sonra indirilir.
Kısa tek dönem testi kârlılık veya istatistiksel kesinlik kanıtı değildir.
Farklı bariyerlerin etiketleri farklıdır; ham F1/AP/BCE tek başına kazanan belirlemez.

**Colab Tesla T4** seçin. Drive: **icinbedava34@gmail.com** hesabını kontrol edin.
Eski arşiv salt okunur. Bu notebook kendi yeni workspace'ini kullanır.
v1.0.0 çalışmasına yeni aday eklemeyin; v1.1.0 için yeni workspace kullanın.
Önce prepare → train → freeze → bundle; calibration raporunu inceleyin.
Test için ayrı hücreyi, eşik/model değiştirmeden çalıştırın. Kesintide aynı ayarlarla devam edin.


## 1 — Çalışma ayarları

Aynı deneye devam ederken WORKSPACE_ID ve sürümü değiştirmeyin.


In [11]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, subprocess, sys

MODE = 'research'  # recovery: yalnızca tablolar; research: yeni deney; archive: finali oku
PHASE = 'basic'
WORKSPACE_ID = 'barriers_october_v11_001'
EXPERIMENTS = ['B25', 'B11', 'B12', 'B152', 'B22', 'B32']  # Her aday ayrı çalıştırılabilir; freeze tümünü ister.
ALLOW_TRAINING = False
RELEASE_TAG = 'advisor-barriers-v1.1.0'  # Sabit yayın; dalın güncel ucunu otomatik çekmez.
REPO_URL = 'https://github.com/umutergul74/YeniBotDeney.git'
REPO_DIR = Path('/content/advisor_barriers_v11_repo')
RUNTIME_DIR = Path('/content/advisor_barriers_v11_runtime')
DRIVE_BASE = Path('/content/drive/MyDrive/yeniBot/advisor_experiments')
SESSION = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
assert WORKSPACE_ID.replace('_','').replace('-','').isalnum()
assert MODE in ('recovery','research','archive')
assert PHASE != 'final' or MODE == 'archive', 'Görülmüş final testi yeniden çalıştırılmaz.'

ALLOW_TEST = True


## 2 — Drive ve ayrı çıktı alanı

Hazırlanmış tablolar, ağırlıklar ve ilerleme kalıcı alana yazılır.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
LEGACY_ROOT = DRIVE_BASE / 'colab_v1'
namespace = {'recovery':'recovery_v2','research':'barrier_research_v1','archive':'archive_audit_v2'}[MODE]
ROOT = DRIVE_BASE / namespace / WORKSPACE_ID
ROOT.mkdir(parents=True,exist_ok=True)
LOG_DIR = ROOT / 'sessions' / SESSION
LOG_DIR.mkdir(parents=True,exist_ok=True)
print('Eski arşiv (salt okunur):',LEGACY_ROOT)
print('Bu çalışmanın kalıcı alanı:',ROOT)
print('Oturum günlükleri:',LOG_DIR)


Mounted at /content/drive
Eski arşiv (salt okunur): /content/drive/MyDrive/yeniBot/advisor_experiments/colab_v1
Bu çalışmanın kalıcı alanı: /content/drive/MyDrive/yeniBot/advisor_experiments/barrier_research_v1/barriers_october_v11_001
Oturum günlükleri: /content/drive/MyDrive/yeniBot/advisor_experiments/barrier_research_v1/barriers_october_v11_001/sessions/20261015T203652345005Z


## 3 — Sabit Git yayını

Yerel değişiklik varsa otomatik silinmez; yanlış sürümle devam edilmez.


In [3]:
if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','--branch',RELEASE_TAG,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
def git(*args):
    return subprocess.check_output(['git','-C',str(REPO_DIR),*args],text=True).strip()
assert not git('status','--porcelain','--untracked-files=no'), 'Kodda yerel değişiklik var.'
assert git('rev-parse','HEAD') == git('rev-parse',RELEASE_TAG+'^{commit}'), 'Yanlış yayın; yeni runtime açın.'
print('Kod sürümü:',git('rev-parse','HEAD'))

assert git('remote','get-url','origin') == REPO_URL, 'Yanlış depo'


Kod sürümü: a9d0d4395d46e9dcc7be00d1160d2582298b2ac3


## 4 — Kilitli araştırma ortamı

Sistem Python'una pip install yapılmaz. Ayrı Python + venv kullanılır; bütün
bağımlılıklar sürüm ve SHA ile sabittir. Drive'daki indirme önbelleği tekrar kullanılır.
uv kilitleri ve paket onbellegi /content altinda tutulur. Drive kalici ciktilari
ve dogrulanmis kurucu dosyasini saklar. Yeni runtime paketleri yeniden indirebilir.
Kurulum başarısızsa sonraki hücreye geçmeyin. Eğitim için Tesla T4 seçin.


In [4]:
def logged(command, name):
    with (LOG_DIR/name).open('a',encoding='utf-8',buffering=1) as log:
        env=dict(os.environ)
        for key in ('PYTHONPATH','PYTHONHOME','PYTHONSTARTUP','IPYTHONDIR','JUPYTER_CONFIG_DIR'):
            env.pop(key,None)
        env['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
        env['MPLBACKEND']='Agg'  # Colab inline backend is host-only.
        with subprocess.Popen(command,cwd=REPO_DIR,env=env,stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line,end=''); log.write(line)
                code=process.wait()
            except BaseException:
                process.terminate(); process.wait(); raise
            if code: raise subprocess.CalledProcessError(code,command)
RECEIPT=RUNTIME_DIR/'environment_receipt.json'
logged([sys.executable,'-I',str(REPO_DIR/'scripts/bootstrap_advisor.py'),
        '--repo',str(REPO_DIR),'--runtime',str(RUNTIME_DIR),
        '--cache',str(DRIVE_BASE/'reproducibility_cache_v2'),
        '--receipt',str(RECEIPT)],'environment_setup.log')
runtime=json.loads(RECEIPT.read_text())
RESEARCH_PYTHON=runtime['python']
print('İzole Python:',RESEARCH_PYTHON)
print('Paket kilidi:',runtime['lock_sha256'])


def execute_step(action):
    command = [RESEARCH_PYTHON, '-I', str(REPO_DIR/'scripts/advisor_barrier_training.py'), action,
               '--root', str(ROOT), '--legacy', str(LEGACY_ROOT), '--session', SESSION,
               '--candidates', *EXPERIMENTS]
    if ALLOW_TRAINING: command.append('--allow-training')
    if ALLOW_TEST: command.append('--allow-test')
    logged(command, action+'.log')


Installed Python 3.13.15 in 1.34s
 + cpython-3.13.15-linux-x86_64-gnu (python3.13)
Using CPython 3.13.15 interpreter at: /content/advisor_barriers_v11_runtime/python/cpython-3.13.15-linux-x86_64-gnu/bin/python3.13
Creating virtual environment at: /content/advisor_barriers_v11_runtime/venv-ff209122c69461b9
Activate with: source /content/advisor_barriers_v11_runtime/venv-ff209122c69461b9/bin/activate
Using Python 3.13.15 environment at: /content/advisor_barriers_v11_runtime/venv-ff209122c69461b9
Resolved 66 packages in 450ms
Prepared 66 packages in 1m 07s
Installed 66 packages in 36.52s
 + certifi==2026.7.22
 + charset-normalizer==3.5.2
 + cloudpickle==3.1.2
 + contourpy==1.4.0
 + cuda-bindings==13.4.4
 + cuda-pathfinder==1.8.3
 + cuda-toolkit==13.0.2
 + cycler==0.12.1
 + filelock==4.2.0
 + fonttools==4.67.0
 + fsspec==2026.9.0
 + hmmlearn==0.3.3
 + idna==3.20
 + iniconfig==2.3.1
 + jinja2==3.1.6
 + joblib==1.6.0
 + kiwisolver==1.5.1
 + llvmlite==0.44.0
 + markupsafe==3.0.4
 + matplotlib

## 5 · Hazırlık

Mevcut fit_raw dosyasının SHA kontrolü; Ekim verisi okunmaz. Eksik arşivde durur.


In [5]:
import os
os.environ["MPLBACKEND"] = "Agg"

In [6]:
execute_step('prepare')


6 prepared frames saved; no test data read.


## 6 · Eğitim

ALLOW_TRAINING=True yapın. Her epoch checkpoint Drive’da. Aynı workspace’e ikinci oturum yazdırmayın.


In [7]:
if ALLOW_TRAINING:
    execute_step('train')
else:
    print('Eğitim kapalı; ayarlarda ALLOW_TRAINING=True seçin.')


fold=0 seed=42 epoch=1 train_BCE=0.606053 val_BCE=0.637550 best=1
fold=0 seed=42 epoch=2 train_BCE=0.596945 val_BCE=0.643941 best=1
fold=0 seed=42 epoch=3 train_BCE=0.591713 val_BCE=0.634980 best=3
fold=0 seed=42 epoch=4 train_BCE=0.585738 val_BCE=0.629835 best=4
fold=0 seed=42 epoch=5 train_BCE=0.582774 val_BCE=0.623354 best=5
fold=0 seed=42 epoch=6 train_BCE=0.575880 val_BCE=0.640496 best=5
fold=0 seed=42 epoch=7 train_BCE=0.567780 val_BCE=0.625695 best=5
fold=0 seed=42 epoch=8 train_BCE=0.561408 val_BCE=0.640952 best=5
fold=0 seed=42 epoch=9 train_BCE=0.555952 val_BCE=0.630188 best=5
fold=0 seed=42 epoch=10 train_BCE=0.544545 val_BCE=0.659182 best=5
fold=0 seed=42 epoch=11 train_BCE=0.544199 val_BCE=0.634543 best=5
fold=0 seed=42 epoch=12 train_BCE=0.531495 val_BCE=0.653658 best=5
fold=0 seed=42 epoch=13 train_BCE=0.519805 val_BCE=0.639723 best=5
fold=0 seed=42 epoch=14 train_BCE=0.514361 val_BCE=0.627010 best=5
fold=0 seed=42 epoch=15 train_BCE=0.503579 val_BCE=0.700945 best=5
fold

## 7 · Eşik seçimi ve değişmez model kilidi

18 eğitim tamamlanmalıdır. Eşik: 0–1, adım 0.005; üç seed ortalama F1. Eşitlikte yüksek eşik. Aday sıralaması: kalibrasyon BCE skill; testten önce kaydedilir.


In [8]:
if ALLOW_TRAINING:
    execute_step('freeze')
else:
    print('Eğitim ve kilitleme henüz başlatılmadı.')


All models and thresholds frozen. Selected on calibration: B11


## 8 · Rapor ve yedek

Küçük ZIP modelleri içermez. Tam yedek için workspace’in tamamını koruyun.


In [9]:
execute_step('bundle')
print('Raporlar:', ROOT/'reports')
print('Modeller:', ROOT/'runs')


Review bundle: /content/drive/.shortcut-targets-by-id/1mk72t_tc2XQ7O2cK_xLl5nZI3wJaRPmI/yeniBot/advisor_experiments/barrier_research_v1/barriers_october_v11_001/reports/barrier_review_20261015T203652345005Z_1792097241045245000.zip Full models remain in workspace.
Raporlar: /content/drive/MyDrive/yeniBot/advisor_experiments/barrier_research_v1/barriers_october_v11_001/reports
Modeller: /content/drive/MyDrive/yeniBot/advisor_experiments/barrier_research_v1/barriers_october_v11_001/runs


## 9 · Nihai test — ayrı adım

Önce calibration_REPORT.md ve calibration_thresholds.png dosyalarını inceleyin. ALLOW_TEST=True olduğunda yalnızca kilitli adaylar değerlendirilir. Sonuç kötü olsa da bu testte yeniden ayar yapılmaz.


In [12]:
if ALLOW_TEST:
    execute_step('test')
    execute_step('bundle')
else:
    print('Test kapalı. Önce eğitim ve kilit; sonra yalnızca bu hücreyi çalıştırın.')


Download final 1H chunk: {'start': '2026-08-31T10:00:00+00:00', 'end_exclusive': '2026-09-01T00:00:00+00:00'}
Download final 1H chunk: {'start': '2026-09-01T00:00:00+00:00', 'end_exclusive': '2026-10-01T00:00:00+00:00'}
Download final 1H chunk: {'start': '2026-10-01T00:00:00+00:00', 'end_exclusive': '2026-10-15T10:00:00+00:00'}
Traceback (most recent call last):
  File "/content/advisor_barriers_v11_repo/scripts/advisor_barrier_training.py", line 11, in <module>
    main()
    ~~~~^^
  File "/content/advisor_barriers_v11_repo/yenibot/experiment/advisor_barrier_training.py", line 357, in main
    elif args.action == 'test': test(root)
                                ~~~~^^^^^^
  File "/content/advisor_barriers_v11_repo/yenibot/experiment/advisor_barrier_training.py", line 297, in test
    source = advisor_final.extend_source(root/'inputs/fit_raw/btc_1h.parquet',
                                         root/'inputs/test_raw', end, cfg)
  File "/content/advisor_barriers_v11_repo/yenibot/

CalledProcessError: Command '['/content/advisor_barriers_v11_runtime/venv-ff209122c69461b9/bin/python', '-I', '/content/advisor_barriers_v11_repo/scripts/advisor_barrier_training.py', 'test', '--root', '/content/drive/MyDrive/yeniBot/advisor_experiments/barrier_research_v1/barriers_october_v11_001', '--legacy', '/content/drive/MyDrive/yeniBot/advisor_experiments/colab_v1', '--session', '20261015T205354279613Z', '--candidates', 'B25', 'B11', 'B12', 'B152', 'B22', 'B32', '--allow-test']' returned non-zero exit status 1.